# 02 — KNN : classification supervisée des jardins
**Objectif :** prédire une catégorie humaine `GARDEN_TYPE` depuis les neuf caractéristiques validées.

Le modèle s'entraîne uniquement sur TRAIN. Les métriques sont calculées sur TEST ; VAL reste disponible pour régler k. Ne pas attribuer des catégories à partir de K-Means. Les résultats sur 31 images sont illustratifs.


In [ ]:
from pathlib import Path
import os, json, psycopg, pandas as pd
from dotenv import load_dotenv
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ai-service/plant-ingestion").exists())
load_dotenv(ROOT / ".env", override=False)
load_dotenv(ROOT / ".env.local", override=True)
dsn = os.environ["GARDNAIZE_ML_DATABASE_URL"]
dsn = dsn.replace("postgresql+psycopg://","postgresql://",1)
FEATURES = ("lawn","terrace","pathway","trees","shrubs","fence","low_wall","stairs","slope")
def conn(): return psycopg.connect(dsn)
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier,NearestNeighbors
from sklearn.metrics import accuracy_score,classification_report,ConfusionMatrixDisplay
import matplotlib.pyplot as plt
with conn() as c:
    with c.cursor() as cur:
        cur.execute("""SELECT g.image_id,g.dataset_split,f.feature_values,l.label_value
        FROM horticulture.garden_image_candidate g
        JOIN horticulture.ml_garden_features f USING(image_id)
        JOIN horticulture.ml_garden_label l ON l.image_id=g.image_id AND l.label_type='GARDEN_TYPE'
        WHERE g.review_status='ACCEPTED' AND f.reviewed=true ORDER BY g.image_id""")
        rows=cur.fetchall()
assert rows,"Attribuer les labels avec le notebook 00"
ids=np.array([r[0] for r in rows]); split=np.array([r[1] for r in rows]); y=np.array([r[3] for r in rows])
X=np.array([[np.nan if r[2].get(k) is None else float(r[2][k]) for k in FEATURES] for r in rows])
train=split=="TRAIN"; test=split=="TEST"
assert train.sum()>=3 and test.sum()>0,"Vérifier les splits TRAIN/TEST"
imp=SimpleImputer(strategy="most_frequent",keep_empty_features=True)
Xt=imp.fit_transform(X[train]); Xtest=imp.transform(X[test])
k=min(3,len(Xt))
model=KNeighborsClassifier(n_neighbors=k)
model.fit(Xt,y[train])
pred=model.predict(Xtest)
print("TRAIN",train.sum(),"TEST",test.sum(),"k",k,"Classes TRAIN",sorted(set(y[train])))
print("Accuracy TEST :",accuracy_score(y[test],pred))
print(classification_report(y[test],pred,zero_division=0))
ConfusionMatrixDisplay.from_predictions(y[test],pred,xticks_rotation=30)
plt.show()



In [ ]:
# Montrer les voisins du premier jardin TEST, sans inclure le jardin TEST dans TRAIN.
dist,ix=model.kneighbors(Xtest[:1])
print("Jardin TEST :",ids[test][0],"| vérité :",y[test][0],"| prédiction :",pred[0])
display(pd.DataFrame({"voisin_image_id":ids[train][ix[0]],
                      "distance":dist[0],"classe_humaine":y[train][ix[0]]}))

